# Modulo 7 · Lezione 3 — Regressione

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Regressione lineare semplice e multipla |
| 2 | Metriche di regressione: MAE, MSE, RMSE, R² |
| 3 | Regressione polinomiale |
| 4 | Regolarizzazione: Ridge e Lasso |
| 5 | Diagnostica con i residui |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

np.random.seed(42)
print("Librerie pronte")

---
## 1. Regressione lineare

```
SEMPLICE (1 feature):
  ŷ = β₀ + β₁·x₁

MULTIPLA (n feature):
  ŷ = β₀ + β₁·x₁ + β₂·x₂ + ... + βₙ·xₙ

Il modello impara i β minimizzando:
  MSE = (1/n) · Σ(yᵢ - ŷᵢ)²

Assunzioni della regressione lineare:
  1. Linearità tra X e y
  2. Residui con distribuzione normale
  3. Omoschedasticità (varianza degli errori costante)
  4. Indipendenza dei residui
  5. Assenza di multicollinearità tra le feature
```

In [ ]:
# Dataset: prevedere il consumo di carburante
np.random.seed(42)
n = 300
peso      = np.random.normal(1500, 400, n).clip(600, 3000)
potencia  = np.random.normal(130, 50, n).clip(60, 300)
velocidad = np.random.normal(120, 30, n).clip(60, 200)
consumo   = (0.005*peso + 0.02*potencia + 0.015*velocidad
             + np.random.normal(0, 1.5, n) + 2)

df = pd.DataFrame({'peso':peso, 'potencia':potencia, 'velocidad':velocidad, 'consumo':consumo})

X = df[['peso', 'potencia', 'velocidad']]
y = df['consumo']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_tr, y_tr)

print("=== Regressione Lineare Multipla ===")
print(f"Intercetta (β₀): {modelo.intercept_:.4f}")
for nombre, coef in zip(X.columns, modelo.coef_):
    print(f"  β({nombre}): {coef:.6f}")

print(f"\nFormula appresa:")
print(f"  consumo = {modelo.intercept_:.4f}")
for nombre, coef in zip(X.columns, modelo.coef_):
    print(f"           + {coef:.5f} × {nombre}")

y_pred = modelo.predict(X_te)
print(f"\nR² sul test: {r2_score(y_te, y_pred):.4f}")

---
## 2. Metriche di regressione

```
MAE  (Mean Absolute Error):  (1/n) Σ |yᵢ - ŷᵢ|
  → Errore medio nelle stesse unità di y
  → Robusto agli outlier
  → Facile da interpretare: "il modello sbaglia in media di ±X unità"

MSE  (Mean Squared Error):   (1/n) Σ (yᵢ - ŷᵢ)²
  → Penalizza gli errori grandi più di quelli piccoli
  → Unità al quadrato (meno interpretabile)

RMSE (Root MSE):             √MSE
  → Stesse unità di y
  → Più sensibile agli outlier rispetto al MAE

R²   (Coefficiente di determinazione):  1 - SS_res/SS_tot
  → Proporzione di varianza spiegata dal modello [0, 1]
  → R²=1: predizione perfetta, R²=0: come predire la media
  → Può essere negativo (modello peggiore della media)
  → Dipende dal dataset: non confrontare R² tra dataset diversi
```

In [ ]:
y_pred = modelo.predict(X_te)

mae  = mean_absolute_error(y_te, y_pred)
mse  = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_te, y_pred)

print("=== Metriche sul test ===")
print(f"MAE:  {mae:.4f}  → il modello sbaglia in media di ±{mae:.2f} L/100km")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}  → errore tipico di {rmse:.2f} L/100km")
print(f"R²:   {r2:.4f}   → il modello spiega il {r2*100:.1f}% della varianza")

# Confronto con la baseline (predice sempre la media)
y_baseline = np.full(len(y_te), y_tr.mean())
print(f"\nBaseline (predice media={y_tr.mean():.2f}):")
print(f"  MAE:  {mean_absolute_error(y_te, y_baseline):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_te, y_baseline)):.4f}")
print(f"  R²:   {r2_score(y_te, y_baseline):.4f}")

# Funzione di metriche riutilizzabile
def evaluar_regresion(y_real, y_pred, nombre="Modelo"):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    print(f"{nombre:20} MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

print("\n=== Tabella comparativa ===")
evaluar_regresion(y_te, y_baseline, "Baseline")
evaluar_regresion(y_te, y_pred, "LinearRegression")

---
## 3. Regressione polinomiale

```
Cosa fare quando la relazione non è lineare?

Aggiungiamo feature polinomiali: x → [x, x², x³, ...]

PolynomialFeatures(degree=2) con feature [a, b] genera:
  [1, a, b, a², a·b, b²]

Il modello resta LINEARE nei parametri:
  ŷ = β₀ + β₁a + β₂b + β₃a² + β₄ab + β₅b²

⚠ A grado più alto → più rischio di overfitting
```

In [ ]:
# Relazione non lineare: temperatura vs vendite di gelati
np.random.seed(42)
temp = np.sort(np.random.uniform(5, 40, 150))
# Relazione quadratica: vendite ottimali intorno ai 28°
ventas = -0.5*(temp - 28)**2 + 100 + np.random.normal(0, 8, 150)
ventas = ventas.clip(0)

X_t = temp.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X_t, ventas, test_size=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_rango = np.linspace(5, 40, 200).reshape(-1, 1)

colores = ['steelblue', 'green', 'orange']
for ax, degree, color in zip(axes, [1, 2, 5], colores):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('modelo', LinearRegression())
    ])
    pipe.fit(X_tr, y_tr)
    r2_tr = r2_score(y_tr, pipe.predict(X_tr))
    r2_te = r2_score(y_te, pipe.predict(X_te))

    ax.scatter(X_tr, y_tr, alpha=0.4, color='gray', s=15, label='Train')
    ax.scatter(X_te, y_te, alpha=0.7, color='red', s=20, marker='x', label='Test')
    ax.plot(x_rango, pipe.predict(x_rango), color=color, lw=2)
    ax.set_title(f'Grado {degree}\nR²_train={r2_tr:.3f}, R²_test={r2_te:.3f}')
    ax.set_xlabel('Temperatura (°C)'); ax.set_ylabel('Vendite')
    ax.legend(fontsize=8)

plt.suptitle('Regressione polinomiale: grado vs overfitting', fontsize=11)
plt.tight_layout(); plt.show()
print("Grado 1: sottoadattamento (underfitting) — curva troppo semplice")
print("Grado 2: buon adattamento — cattura la forma quadratica reale")
print("Grado 5: overfitting — si adatta al rumore del train")

---
## 4. Regolarizzazione: Ridge e Lasso

```
PROBLEMA: coefficienti troppo grandi → overfitting
SOLUZIONE: aggiungere una penalità ai coefficienti grandi

RIDGE (L2):  MSE + α · Σβᵢ²
  → Riduce tutti i coefficienti, nessuno arriva esattamente a 0
  → Utile quando tutte le feature sono rilevanti

LASSO (L1):  MSE + α · Σ|βᵢ|
  → Può portare i coefficienti esattamente a 0 → selezione delle feature
  → Utile quando solo alcune feature sono rilevanti

α (alpha): controlla la forza della regolarizzazione
  α=0 → equivale a LinearRegression
  α→∞ → tutti i coefficienti → 0
```

In [ ]:
# Dataset con molte feature, alcune irrilevanti
np.random.seed(42)
n, n_features = 200, 20
X_reg = np.random.randn(n, n_features)
# Solo le prime 5 feature sono rilevanti
coef_real = np.array([3, -2, 1.5, -1, 0.5] + [0]*15)
y_reg = X_reg @ coef_real + np.random.randn(n) * 0.5

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

modelos = {
    'LinearRegression': LinearRegression(),
    'Ridge α=0.1':      Ridge(alpha=0.1),
    'Ridge α=10':       Ridge(alpha=10),
    'Lasso α=0.1':      Lasso(alpha=0.1),
    'Lasso α=1.0':      Lasso(alpha=1.0),
}

print(f"{'Modello':20} {'R²_train':>10} {'R²_test':>10} {'Coef≠0':>8} {'Max|coef|':>12}")
resultados = {}
for nombre, m in modelos.items():
    m.fit(X_tr, y_tr)
    r2_tr = r2_score(y_tr, m.predict(X_tr))
    r2_te = r2_score(y_te, m.predict(X_te))
    n_nonzero = np.sum(np.abs(m.coef_) > 1e-6)
    max_coef  = np.max(np.abs(m.coef_))
    print(f"{nombre:20} {r2_tr:>10.4f} {r2_te:>10.4f} {n_nonzero:>8} {max_coef:>12.4f}")
    resultados[nombre] = m.coef_

# Visualizzare i coefficienti
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

x_idx = np.arange(n_features)
axes[0].bar(x_idx - 0.3, coef_real, 0.3, label='Reale', color='black', alpha=0.7)
axes[0].bar(x_idx,       resultados['LinearRegression'], 0.3, label='LinearReg', color='steelblue', alpha=0.7)
axes[0].bar(x_idx + 0.3, resultados['Ridge α=10'], 0.3, label='Ridge α=10', color='coral', alpha=0.7)
axes[0].set_title('Coefficienti: Reale vs LinearReg vs Ridge')
axes[0].legend(); axes[0].set_xlabel('Feature')

axes[1].bar(x_idx - 0.15, coef_real, 0.3, label='Reale', color='black', alpha=0.7)
axes[1].bar(x_idx + 0.15, resultados['Lasso α=0.1'], 0.3, label='Lasso α=0.1', color='green', alpha=0.7)
axes[1].set_title('Lasso: selezione delle feature (coef=0 → feature eliminata)')
axes[1].legend(); axes[1].set_xlabel('Feature')

plt.tight_layout(); plt.show()
print(f"\nLasso elimina {np.sum(np.abs(resultados['Lasso α=0.1']) < 1e-6)} feature su {n_features}")

---
## 5. Diagnostica con i residui

In [ ]:
# Residui: differenza tra valore reale e predetto
y_pred_tr = modelo.predict(X_tr)
y_pred_te = modelo.predict(X_te)
residuos_tr = y_tr - y_pred_tr
residuos_te = y_te - y_pred_te

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Reale vs Predetto
axes[0,0].scatter(y_pred_te, y_te, alpha=0.5, color='steelblue', s=20)
lim = [min(y_te.min(), y_pred_te.min()), max(y_te.max(), y_pred_te.max())]
axes[0,0].plot(lim, lim, 'r--', lw=2, label='Perfetta')
axes[0,0].set_xlabel('Predetto'); axes[0,0].set_ylabel('Reale')
axes[0,0].set_title('Reale vs Predetto'); axes[0,0].legend()

# 2. Residui vs Predetto
axes[0,1].scatter(y_pred_te, residuos_te, alpha=0.5, color='coral', s=20)
axes[0,1].axhline(0, color='red', ls='--', lw=2)
axes[0,1].set_xlabel('Predetto'); axes[0,1].set_ylabel('Residuo')
axes[0,1].set_title('Residui vs Predetto\n(devono stare intorno a 0)')

# 3. Istogramma dei residui
from scipy import stats
axes[1,0].hist(residuos_te, bins=25, density=True, color='steelblue', alpha=0.7)
x_norm = np.linspace(residuos_te.min(), residuos_te.max(), 100)
axes[1,0].plot(x_norm, stats.norm.pdf(x_norm, residuos_te.mean(), residuos_te.std()), 'r-', lw=2)
axes[1,0].set_title('Distribuzione dei residui\n(devono essere normali)')

# 4. QQ-plot dei residui
stats.probplot(residuos_te, plot=axes[1,1])
axes[1,1].set_title('QQ-plot dei residui')

plt.suptitle('Diagnostica del modello di regressione', fontsize=11)
plt.tight_layout(); plt.show()

# Test di normalità dei residui
_, p_shapiro = stats.shapiro(residuos_te)
print(f"Test Shapiro-Wilk sui residui: p={p_shapiro:.4f}")
print(f"Residui {'normali ✓' if p_shapiro > 0.05 else 'NON normali ✗'} (α=0.05)")
print(f"Media dei residui: {residuos_te.mean():.6f} (deve essere ≈0)")

---
## Esercitazione

### Esercizio 1 — Regressione sul dataset California Housing

**Consegna:** Usa il dataset delle case della California:

```python
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
```

1. Esplora il dataset: shape, feature_names, statistiche
2. Dividi in train/test (80/20)
3. Scala le feature con StandardScaler (ricorda: fit solo sul train)
4. Addestra LinearRegression
5. Calcola MAE, RMSE e R² sul test. Interpreta i risultati

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

housing = fetch_california_housing()
X, y = housing.data, housing.target  # y in $100.000

# 1. Esplorazione
print(f"Shape: {X.shape}")
print(f"Feature: {housing.feature_names}")
print(f"Target: prezzo mediano della casa (×$100k). Intervallo: [{y.min():.2f}, {y.max():.2f}]")

# 2. Split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scalatura
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

# 4. Addestramento
modelo = LinearRegression().fit(X_tr_s, y_tr)

# 5. Valutazione
y_pred = modelo.predict(X_te_s)
mae  = mean_absolute_error(y_te, y_pred)
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2   = r2_score(y_te, y_pred)

print(f"\nMAE:  {mae:.4f} (×$100k) = ${mae*100000:,.0f} di errore medio")
print(f"RMSE: {rmse:.4f} (×$100k) = ${rmse*100000:,.0f} di errore tipico")
print(f"R²:   {r2:.4f} → il modello spiega il {r2*100:.1f}% della varianza")
print("\nCoefficienti più rilevanti:")
for feat, coef in sorted(zip(housing.feature_names, modelo.coef_),
                          key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:20}: {coef:+.4f}")
```
</details>

---
### Esercizio 2 — Trovare il grado polinomiale migliore

**Consegna:** Sull'esempio temperatura-vendite di gelati:

```python
np.random.seed(42)
temp   = np.sort(np.random.uniform(5, 40, 150)).reshape(-1,1)
ventas = (-0.5*(temp.ravel()-28)**2 + 100 + np.random.normal(0, 8, 150)).clip(0)
```

1. Prova i gradi da 1 a 8
2. Per ogni grado calcola R² su train e test
3. Visualizza in un grafico: asse x = grado, asse y = R²
4. Da quale grado in poi inizia l'overfitting?
5. Qual è il grado ottimale? Perché?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

np.random.seed(42)
temp   = np.sort(np.random.uniform(5, 40, 150)).reshape(-1,1)
ventas = (-0.5*(temp.ravel()-28)**2 + 100 + np.random.normal(0, 8, 150)).clip(0)

X_tr, X_te, y_tr, y_te = train_test_split(temp, ventas, test_size=0.2, random_state=42)

r2_trains, r2_tests = [], []
for d in range(1, 9):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(d, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    pipe.fit(X_tr, y_tr)
    r2_trains.append(r2_score(y_tr, pipe.predict(X_tr)))
    r2_tests.append(r2_score(y_te, pipe.predict(X_te)))

grados = range(1, 9)
plt.figure(figsize=(10, 4))
plt.plot(grados, r2_trains, 'b-o', label='R² train')
plt.plot(grados, r2_tests,  'r-o', label='R² test')
plt.axvline(2, color='green', ls='--', alpha=0.7, label='Grado ottimale=2')
plt.xlabel('Grado polinomiale'); plt.ylabel('R²')
plt.title('R² su train e test per grado polinomiale')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

mejor_grado = np.argmax(r2_tests) + 1
print(f"Grado con R² più alto sul test: {mejor_grado}")
print(f"R²_train={r2_trains[mejor_grado-1]:.4f}, R²_test={r2_tests[mejor_grado-1]:.4f}")
print("\nA partire dal grado 3-4, R² train continua a salire ma quello sul test scende → overfitting")
print("Il grado 2 cattura la relazione quadratica reale. Più grado = adattarsi al rumore.")
```
</details>

---
### Esercizio 3 — Ridge vs Lasso: quando usare ciascuno?

**Consegna:** Genera due dataset:

```python
np.random.seed(42)
n, p = 100, 30
X = np.random.randn(n, p)

# Dataset A: tutte le feature contano (coefficienti piccoli)
coef_A = np.random.randn(p) * 0.5
y_A = X @ coef_A + np.random.randn(n)

# Dataset B: solo 5 feature contano (il resto è rumore)
coef_B = np.zeros(p)
coef_B[:5] = np.array([3, -2, 1.5, -1, 0.5])
y_B = X @ coef_B + np.random.randn(n)
```

Per ogni dataset, confronta Ridge α=1 e Lasso α=0.1 con LinearRegression. Quale modello vince in ciascun caso? Perché?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

np.random.seed(42)
n, p = 100, 30
X = np.random.randn(n, p)

coef_A = np.random.randn(p) * 0.5
y_A = X @ coef_A + np.random.randn(n)

coef_B = np.zeros(p)
coef_B[:5] = [3, -2, 1.5, -1, 0.5]
y_B = X @ coef_B + np.random.randn(n)

modelos = {
    'LinearRegression': LinearRegression(),
    'Ridge α=1':        Ridge(alpha=1),
    'Lasso α=0.1':      Lasso(alpha=0.1),
}

for nombre_ds, (X_ds, y_ds) in [('Dataset A (tutte contano)', (X, y_A)),
                                  ('Dataset B (solo 5)',        (X, y_B))]:
    print(f"\n=== {nombre_ds} ===")
    X_tr, X_te, y_tr, y_te = train_test_split(X_ds, y_ds, test_size=0.3, random_state=42)
    for nombre_m, m in modelos.items():
        m.fit(X_tr, y_tr)
        r2_tr = r2_score(y_tr, m.predict(X_tr))
        r2_te = r2_score(y_te, m.predict(X_te))
        n_cero = np.sum(np.abs(m.coef_) < 1e-6)
        print(f"  {nombre_m:20} R²_tr={r2_tr:.3f} R²_te={r2_te:.3f} coef_zero={n_cero}")

print("\nConclusione:")
print("  Dataset A: Ridge vince — tutti i coefficienti sono piccoli ma non zero.")
print("  Dataset B: Lasso vince — porta esattamente a 0 le 25 feature irrilevanti.")
```
</details>

---
### Esercizio 4 — Diagnostica dei residui

**Consegna:** Genera un dataset in cui LinearRegression ha dei problemi (eteroschedasticità):

```python
np.random.seed(42)
x = np.sort(np.random.uniform(1, 100, 200))
# L'errore cresce con x (eteroschedasticità)
y = 2*x + np.random.randn(200) * x * 0.2
```

1. Addestra LinearRegression e calcola R²
2. Fai la diagnostica dei residui (4 grafici)
3. Che problema individui?
4. Applica la trasformazione log: `y_log = np.log(y)`, riaddestra e confronta i residui

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

np.random.seed(42)
x = np.sort(np.random.uniform(1, 100, 200))
y = 2*x + np.random.randn(200) * x * 0.2

def diagnostico(X, y, titulo):
    X_tr, X_te, y_tr, y_te = train_test_split(X.reshape(-1,1), y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(X_tr, y_tr)
    y_pred = m.predict(X_te)
    res = y_te - y_pred
    r2  = r2_score(y_te, y_pred)

    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    fig.suptitle(f'{titulo} (R²={r2:.4f})')

    axes[0].scatter(y_pred, res, alpha=0.5, s=15, color='steelblue')
    axes[0].axhline(0, color='red', ls='--'); axes[0].set_title('Residui vs Predetto')

    axes[1].hist(res, bins=30, color='coral', density=True)
    xn = np.linspace(res.min(), res.max(), 100)
    axes[1].plot(xn, stats.norm.pdf(xn, res.mean(), res.std()), 'b-', lw=2)
    axes[1].set_title('Distribuzione dei residui')

    stats.probplot(res, plot=axes[2])
    axes[2].set_title('QQ-plot')

    plt.tight_layout(); plt.show()
    return r2

r2_orig = diagnostico(x, y, 'Con y originale (eteroschedasticità)')
r2_log  = diagnostico(x, np.log(y), 'Con log(y) (residui più omogenei)')

print(f"R² originale: {r2_orig:.4f}")
print(f"R² con log:  {r2_log:.4f}")
print("\nIl QQ-plot e il grafico dei residui vs predetto mostrano:")
print("  Originale: la varianza degli errori CRESCE con x (eteroschedasticità)")
print("  log(y): residui più uniformi → assunzione di omoschedasticità rispettata")
```
</details>

---
### Esercizio 5 (sfida) — Selezione dell'α ottimale con validazione

**Consegna:** Implementa una ricerca manuale del miglior alpha per Ridge e Lasso:

1. Genera un dataset con 15 feature (5 rilevanti, 10 di rumore)
2. Prova gli alpha: `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`
3. Per ogni alpha, calcola R² su train e test
4. Visualizza le curve di Ridge e Lasso
5. Qual è l'alpha ottimale per ciascun modello? Perché divergono?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

np.random.seed(42)
n, p = 150, 15
X = np.random.randn(n, p)
coef = np.array([2, -1.5, 1, -0.8, 0.5] + [0]*10)
y = X @ coef + np.random.randn(n) * 0.5

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, ModelClass, nombre in zip(axes, [Ridge, Lasso], ['Ridge', 'Lasso']):
    r2_tr_list, r2_te_list = [], []
    for alpha in alphas:
        m = ModelClass(alpha=alpha).fit(X_tr, y_tr)
        r2_tr_list.append(r2_score(y_tr, m.predict(X_tr)))
        r2_te_list.append(r2_score(y_te, m.predict(X_te)))

    mejor_idx = np.argmax(r2_te_list)
    ax.semilogx(alphas, r2_tr_list, 'b-o', label='R² train')
    ax.semilogx(alphas, r2_te_list, 'r-o', label='R² test')
    ax.axvline(alphas[mejor_idx], color='green', ls='--',
               label=f'α ottimale={alphas[mejor_idx]}')
    ax.set_xlabel('Alpha (log)'); ax.set_ylabel('R²')
    ax.set_title(f'{nombre}: R² vs Alpha')
    ax.legend(); ax.grid(True, alpha=0.3)
    print(f"{nombre}: alpha migliore={alphas[mejor_idx]}, R²_test={r2_te_list[mejor_idx]:.4f}")

plt.tight_layout(); plt.show()
print("\nConclusione:")
print("  Lasso ha bisogno di un alpha minore per non eliminare i coefficienti rilevanti.")
print("  Ridge può tollerare alpha maggiori — riduce soltanto, non elimina.")
print("  Nella C7 vedremo GridSearchCV per trovare l'alpha in modo automatico.")
```
</details>

---
## Riepilogo

```python
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Metriche di regressione
mae  = mean_absolute_error(y_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_real, y_pred))
r2   = r2_score(y_real, y_pred)

# Regressione polinomiale
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_train)

# Regolarizzazione
Ridge(alpha=1.0)   # L2: riduce i coefficienti, non li annulla
Lasso(alpha=0.1)   # L1: può annullare i coefficienti (selezione)
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 4** passiamo alla classificazione con la Regressione Logistica e impariamo le metriche specifiche per i problemi di classificazione.